## Environment Status and Setup
*Note that venv is used instead of conda for development

In [2]:
# Check CPU status (core and version) and RAM size
!lscpu | grep -E 'Model name|CPU\(s\)'
!echo ""
!free -h

CPU(s):                                  32
On-line CPU(s) list:                     0-31
Model name:                              13th Gen Intel(R) Core(TM) i9-13900K
CPU(s) scaling MHz:                      62%
NUMA node0 CPU(s):                       0-31

               total        used        free      shared  buff/cache   available
Mem:            62Gi       5.6Gi        49Gi        53Mi       7.9Gi        56Gi
Swap:          8.0Gi       2.8Gi       5.2Gi


# Reminder
## Note that the model is trained using 60GB of RAM; consider a bigger machine, else memory crash may occur.

In [3]:
%%bash
# Create a virtual environment using Python 3.12.3
python3.12 -m venv .venv

# Activate the virtual environment and install ipykernel
source .venv/bin/activate
pip install --upgrade pip
pip install ipykernel

# Register the virtual environment as a Jupyter kernel
python -m ipykernel install --user --name=venv-3.12 --display-name="Python 3.12.3 (venv)"


Installed kernelspec venv-3.12 in /home/vizlab/.local/share/jupyter/kernels/venv-3.12


In [4]:
# Install all required dependencies for the environment
import sys
!{sys.executable} -m pip install ipykernel numpy pandas matplotlib tqdm opencv-python scikit-image scikit-learn scikit-learn-intelex lightgbm imbalanced-learn

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached scikit_image-0.26.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (15 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached scikit_learn_intelex-2025.11.0-py312-none-manylinux_2_28_x86_64.whl.metadata (9.9 kB)
  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manyl

### Custom SVM Training Pipeline
Orchestrating data loading, image augmentations (flips, shifts, rotations), and direct-to-disk caching to manage memory efficiently.
*Note that code is assisted partially with help of LLMs.

In [5]:
# Install requirements for our new augmentation methodology
# !pip install scikit-learn-intelex imbalanced-learn

from sklearnex import patch_sklearn
patch_sklearn()

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
from tqdm import tqdm
import cv2

class DataLoader:
    """Handles loading raw images and paths from CSV."""
    def __init__(self, base_dir="."):
        self.base_dir = base_dir
        
    def process_images(self, img_names, folder):
        images = []
        for name in tqdm(img_names, desc=f"Loading {folder}"):
            path = os.path.join(self.base_dir, folder, name)
            # plt.imread loads RGB. Normalize to 0-255 uint8 if float
            img = plt.imread(path)
            if img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            images.append(img)
        return np.array(images)

    def load_train_data(self):
        df = pd.read_csv(os.path.join(self.base_dir, "train.csv"))
        X = self.process_images(df['im_name'].values, "train_ims")
        y = df['label'].values
        return X, y
        
    def load_test_data(self):
        df = pd.read_csv(os.path.join(self.base_dir, "test.csv"))
        X = self.process_images(df['im_name'].values, "test_ims")
        return X, df

class ImageAugmenter:
    """Applies data augmentation dynamically and saves directly to disk to save memory."""
    def __init__(self, shifts=4, cache_dir="./temp/data"):
        self.shifts = shifts
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)
        
    def augment_and_save(self, X):
        print("Saving base image augmentations directly to disk...")
        
        # paths = ["Original", "flip", "shift_2", "shift_up_2"]
        paths = ["Original", "flip", "shift_4", "shift_up_4", "rotate_10"]
        
        # 1. Original
        if not os.path.exists(f"{self.cache_dir}/Original.npy"):
            np.save(f"{self.cache_dir}/Original.npy", X)
        
        # 2. Horizontal Flip
        if not os.path.exists(f"{self.cache_dir}/flip.npy"):
            flips = np.array([np.flip(img, axis=1) for img in tqdm(X, desc="Flipping")])
            np.save(f"{self.cache_dir}/flip.npy", flips)
            del flips # clear memory
        
        # 3. Shift Right (4 pixels)
        # if not os.path.exists(f"{self.cache_dir}/shift_2.npy"):
        #     shifts_r = np.array([np.roll(img, shift=self.shifts, axis=1) for img in tqdm(X, desc="Shifting Right")])
        #     np.save(f"{self.cache_dir}/shift_2.npy", shifts_r)
        #     del shifts_r
        if not os.path.exists(f"{self.cache_dir}/shift_4.npy"):
             shifts_r = np.array([np.roll(img, shift=self.shifts, axis=1) for img in tqdm(X, desc="Shifting Right")])
             np.save(f"{self.cache_dir}/shift_4.npy", shifts_r)
             del shifts_r
        
        # 4. Shift Up (4 pixels)
        # if not os.path.exists(f"{self.cache_dir}/shift_up_2.npy"):
        #     shifts_u = np.array([np.roll(img, shift=-self.shifts, axis=0) for img in tqdm(X, desc="Shifting Up")])
        #     np.save(f"{self.cache_dir}/shift_up_2.npy", shifts_u)
        #     del shifts_u
        if not os.path.exists(f"{self.cache_dir}/shift_up_4.npy"):
             shifts_u = np.array([np.roll(img, shift=-self.shifts, axis=0) for img in tqdm(X, desc="Shifting Up")])
             np.save(f"{self.cache_dir}/shift_up_4.npy", shifts_u)
             del shifts_u
             
        # 5. Rotation (-10 to 10 degrees approximation / or 5 deg) using pure cv2
        if not os.path.exists(f"{self.cache_dir}/rotate_10.npy"):
            rotated = []
            for img in tqdm(X, desc="Rotating 10 deg"):
                # Rotate around center
                (h, w) = img.shape[:2]
                center = (w // 2, h // 2)
                M = cv2.getRotationMatrix2D(center, 10, 1.0) # 10 degrees
                rot = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
                rotated.append(rot)
            np.save(f"{self.cache_dir}/rotate_10.npy", np.array(rotated))
            del rotated

# Prepare directories and cache raw dataset arrays + augmentations
os.makedirs("./temp/data", exist_ok=True)

if not os.path.exists("./temp/data/Original.npy") or not os.path.exists("./temp/labels.npy"):
    print("Caching training data...")
    loader = DataLoader()
    X_train_raw, y_train_raw = loader.load_train_data()
    np.save("./temp/labels.npy", y_train_raw)
    
    augmenter = ImageAugmenter()
    augmenter.augment_and_save(X_train_raw)
    print("Train caching & augmentation complete!")
else:
    print("Train cache found, checking augmentations...")
    # Ensure augmentations exist even if original is loaded
    expected = ["flip", "shift_4", "shift_up_4", "rotate_10"]
    if not all(os.path.exists(f"./temp/data/{aug}.npy") for aug in expected):
    # if not all(os.path.exists(f"./temp/data/{aug}.npy") for aug in ["flip", "shift_4", "shift_up_4"]):
        X_train_raw = np.load("./temp/data/Original.npy")
        augmenter = ImageAugmenter()
        augmenter.augment_and_save(X_train_raw)
    else:
         print("All augmentations already cached!")

if not os.path.exists("./temp/data/pre_X_test.npy"):
    print("Caching test data...")
    loader = DataLoader()
    X_test_raw, _ = loader.load_test_data()
    np.save("./temp/data/pre_X_test.npy", X_test_raw)
    print("Test caching complete!")
else:
    print("Test cache found, skipping raw load.")

Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


Caching training data...


Loading train_ims: 100%|██████████| 50000/50000 [00:02<00:00, 18122.53it/s]


Saving base image augmentations directly to disk...


Rotating 10 deg: 100%|██████████| 50000/50000 [00:00<00:00, 59351.58it/s]


Train caching & augmentation complete!
Caching test data...


Loading test_ims: 100%|██████████| 10000/10000 [00:00<00:00, 18502.92it/s]

Test caching complete!


### 1. Dynamic Feature Extraction
Generating and caching local binary patterns (LBP), histograms of oriented gradients (HOG), and color histograms on the fly for each augmented variant.

In [6]:
import os
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern, hog, graycomatrix, graycoprops
import cv2

def compute_missing_feature(aug, feat, base_path):
    print(f"Feature {feat} missing for {aug}. Generating on the fly...")
    data_path = f"./temp/data/{aug}.npy"
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Cannot generate feature {feat}, original image data {data_path} is missing.")
    
    images = np.load(data_path)
    out_path = f"{base_path}/{aug}/{feat}.npy"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    
    feature_list = []
    
    for img in tqdm(images, desc=f"Generating {feat} for {aug}"):
        # Convert HWC or CHW to HWC if necessary
        if img.shape[0] == 3:
            img = np.transpose(img, (1, 2, 0))
            
        if len(img.shape) == 3 and img.shape[-1] == 3:
            # Ensure float type for color conversion if it's in 0-1 range
            if img.dtype != np.uint8 and img.max() <= 1.0:
                img = (img * 255).astype(np.uint8)
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        else:
            gray = img
            
        if gray.max() <= 1.0:
            gray = (gray * 255).astype(np.uint8)
        elif gray.dtype != np.uint8:
            gray = gray.astype(np.uint8)

        # if feat.startswith("lbp_"):
        #     # Format expected: lbp_radius_npoints
        #     parts = feat.split("_")
        #     radius = int(parts[1])
        #     n_points = int(parts[2])
        #     lbp = local_binary_pattern(gray, n_points, radius, method="uniform")
        #     (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
        #     hist = hist.astype("float")
        #     hist /= (hist.sum() + 1e-7)
        #     feature_list.append(hist)

        if feat.startswith("lbp_"):
            parts = feat.split("_")
            radius = int(parts[1])
            n_points = int(parts[2])
            lbp = local_binary_pattern(gray, n_points, radius, method="uniform")
            (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, n_points + 3), range=(0, n_points + 2))
            hist = hist.astype("float")
            hist /= (hist.sum() + 1e-7)
            feature_list.append(hist)
            
        elif feat == "hu_moments":
            moments = cv2.moments(gray)
            hu = cv2.HuMoments(moments).flatten()
            hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
            feature_list.append(hu)
            
        # elif feat == "haralick":
        #     # Using skimage GLCM (Gray-Level Co-occurrence Matrix) to approximate Haralick features
        #     # Alternatively, if you have `mahotas` installed, you could use `mahotas.features.haralick(gray).mean(axis=0)`
        #     distances = [1, 2, 3]
        #     angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
        #     # Ensure bins are correct for 8-bit image
        #     glcm = graycomatrix(gray, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)
            
        #     haralick_feats = []
        #     for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']:
        #         haralick_feats.extend(graycoprops(glcm, prop).flatten())
        #     feature_list.append(haralick_feats)
            
        elif feat.startswith("hog_"):
            # Parse settings from feature name like "hog_9_(4, 4)_(2, 2)_2"
            parts = feat.replace("(", "").replace(")", "").replace(" ", "").split("_")
            orientations = int(parts[1])
            ppc_val = int(parts[2].split(",")[0])
            cpb_val = int(parts[4].split(",")[0])
            
            # Using channel_axis=-1 requires color image
            color_img = img if len(img.shape) == 3 and img.shape[-1] == 3 else cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            
            fd = hog(color_img, orientations=orientations, pixels_per_cell=(ppc_val, ppc_val),
                     cells_per_block=(cpb_val, cpb_val), channel_axis=-1)
            feature_list.append(fd)
            
        elif feat == "raw":
            feature_list.append(gray.flatten())
            
        elif feat == "raw_color":
            color_img = img if len(img.shape) == 3 and img.shape[-1] == 3 else cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            feature_list.append(color_img.flatten())
            
        elif feat == "color_hist":
            color_img = img if len(img.shape) == 3 and img.shape[-1] == 3 else cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
            hsv = cv2.cvtColor(color_img, cv2.COLOR_RGB2HSV)
            # Create a 3D histogram using 8 bins per color channel
            hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
            cv2.normalize(hist, hist)
            feature_list.append(hist.flatten())

        # elif feat == "color_hist":
        #     color_img = img if len(img.shape) == 3 and img.shape[-1] == 3 else cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
        #     hsv = cv2.cvtColor(color_img, cv2.COLOR_RGB2HSV)
            
        #     # Better binning for HSV (more perceptually meaningful)
        #     hist = cv2.calcHist([hsv], [0, 1, 2], None, [12, 8, 8], [0, 180, 0, 256, 0, 256])  # Hue has 180 range
        #     cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_L1)
        #     feature_list.append(hist.flatten())
            
        # else:
        #     raise NotImplementedError(f"On-the-fly generation for {feat} is not implemented.")
            
    result = np.array(feature_list)
    np.save(out_path, result)
    return result

def load_cached_feature_matrix(aug_names, feature_names, base_path="./temp/feature"):
    loaded_features = []
    for feat in feature_names:
        aug_matrices = []
        for aug in aug_names:
            file_path = f"{base_path}/{aug}/{feat}.npy"
            if not os.path.exists(file_path):
                # Generates missing feature array automatically 
                aug_matrix = compute_missing_feature(aug, feat, base_path)
                aug_matrices.append(aug_matrix)
            else:
                aug_matrices.append(np.load(file_path))
        
        # Combine all augmentations vertically for this specific feature
        combined_augs = np.vstack(aug_matrices)
        
        # Flatten to 2D if needed: (N, features)
        combined_augs = combined_augs.reshape(combined_augs.shape[0], -1)
        loaded_features.append(combined_augs)
        
    # Concatenate horizontally (feature union)
    return np.hstack(loaded_features)

# Define the exact augmentations used for training
# augmentation_variants = ["Original", "shift_2", "shift_up_2", "flip"]
augmentation_variants = ["Original", "shift_4", "shift_up_4", "flip"]
# augmentation_variants = ["Original", "shift_4", "flip", "rotate_10"]

# # Define the exact available feature files included here with those ready to optionally compute
# selected_features = [
#     "hog_9_(6, 6)_(2, 2)_2",
#     "hog_9_(4, 4)_(2, 2)_2",
#     "hog_9_(6, 6)_(2, 2)_2",
#     "hog_9_(8, 8)_(2, 2)_2",
#     "hog_18_(8, 8)_(2, 2)_2",
#     "hog_9_(10, 10)_(2, 2)_2",
#     "hog_18_(10, 10)_(2, 2)_2",
#     "lbp_1_8",
#     "hu_moments",
#     "raw",
#     "raw_color",
#     "color_hist"
# ]

# selected_features = [
#     "hog_9_(4, 4)_(2, 2)_2",      # finest details → best for animals
#     # "hog_18_(4,4)_(2, 2)_2",
#     "hog_9_(6, 6)_(2, 2)_2",
#     "hog_9_(8, 8)_(2, 2)_2",
#     "hog_18_(8, 8)_(2, 2)_2",     # more orientations for shape
#     "hog_9_(10, 10)_(2, 2)_2",
#     "lbp_1_8",
#     "lbp_2_16",
#     # "haralick",         
#     # "hu_moments",
#     "color_hist",
#     "raw_color"
#     # "raw"
# ]

selected_features = [
    "hog_9_(4, 4)_(2, 2)_2",      # finest details → best for animals
    "hog_9_(6, 6)_(2, 2)_2",
    "hog_9_(8, 8)_(2, 2)_2",
    "hog_18_(8, 8)_(2, 2)_2",     # more orientations for shape
    "hog_9_(10, 10)_(2, 2)_2",
    "lbp_1_8",
    "lbp_2_16",
    "color_hist",
    "raw_color"
]

### 2. Constructing the Unified Feature Matrix
Loading the cached features and concatenating them. The base labels are duplicated (tiled) to match the dimensions of the expanded augmented datasets.

In [7]:
print("Reading Feature Matrices...")

# Load baseline labels
base_labels = np.load("./temp/labels.npy")

# Match label rows to duplicated augmentation variants
y_loaded = np.tile(base_labels, len(augmentation_variants))

# Pull flat feature vectors
X_loaded = load_cached_feature_matrix(augmentation_variants, selected_features)

print(f"Prepared Feature Dataset X={X_loaded.shape}, y={y_loaded.shape}")

Reading Feature Matrices...
Feature hog_9_(4, 4)_(2, 2)_2 missing for Original. Generating on the fly...


Generating hog_9_(4, 4)_(2, 2)_2 for Original: 100%|██████████| 50000/50000 [00:19<00:00, 2504.29it/s]


Feature hog_9_(4, 4)_(2, 2)_2 missing for shift_4. Generating on the fly...


Generating hog_9_(4, 4)_(2, 2)_2 for shift_4: 100%|██████████| 50000/50000 [00:19<00:00, 2533.21it/s]


Feature hog_9_(4, 4)_(2, 2)_2 missing for shift_up_4. Generating on the fly...


...

Feature raw_color missing for shift_up_4. Generating on the fly...


Generating raw_color for shift_up_4: 100%|██████████| 50000/50000 [00:00<00:00, 325323.94it/s]


Feature raw_color missing for flip. Generating on the fly...


Generating raw_color for flip: 100%|██████████| 50000/50000 [00:00<00:00, 324147.30it/s]


Prepared Feature Dataset X=(200000, 7068), y=(200000,)


### 3. Model Definition and Fitting
Building a sequential `scikit-learn` pipeline with `StandardScaler`, `PCA` for dimensionality reduction, and `SVC` (Support Vector Classifier). The model is trained on the entirely aggregated dataset.

In [8]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
import numpy as np
import lightgbm as lgb

""" # Leakage 
X_train_cv, X_val_cv, y_train_cv, y_val_cv = train_test_split(
    X_loaded, y_loaded, test_size=0.2, random_state=123, stratify=y_loaded
) """

""" # Train on ALL 200k (50k * 4) images
X_train_cv = X_loaded
y_train_cv = y_loaded

print(f"Training split: X={X_train_cv.shape}, y={y_train_cv.shape}")

 """
# ORIGINAL CODE (Validation split & Leakage fix): 
# FIX LEAKAGE: Split on the core/base image indices first to avoid validation images creeping into training via augmentations
num_base_images = len(base_labels)
base_indices = np.arange(num_base_images)

train_base_idx, val_base_idx = train_test_split(
    base_indices, test_size=0.2, random_state=123
)

# Expand the training indices to include all augmented versions of the base training images
num_augs = len(augmentation_variants)
train_full_idx = [idx + i * num_base_images for i in range(num_augs) for idx in train_base_idx]

# VALIDATION LEAKAGE FIX #2: Evaluate only on the Original untouched images!
# Since 'Original' is the 0th augmentation layout, we just grab the base validation indices. 
# val_base_only_idx = val_base_idx
# val_full_idx = [idx + i * num_base_images for i in range(num_augs) for idx in val_base_idx]
val_full_idx = val_base_idx


X_train_cv = X_loaded[train_full_idx]
y_train_cv = y_loaded[train_full_idx]

X_val_cv = X_loaded[val_full_idx]
y_val_cv = y_loaded[val_full_idx]


# ==========================================
# Build sequential modeling pipeline cleanly
# ==========================================

# --- OPTION 1: Support Vector Machine (SVM) ---
classifier_chain = make_pipeline(
    StandardScaler(copy=False),
    PCA(n_components=0.85, random_state=42, copy=False),
    SVC(kernel='rbf', C=10, class_weight='balanced', cache_size=100, random_state=42)
)

# # --- OPTION 2: LightGBM ---
# lgbm_params = {
#     'learning_rate': 0.06160521634512509, 
#     'num_leaves': 57, 
#     'n_estimators': 2330, 
#     'subsample': 0.790468150125655, 
#     'colsample_bytree': 0.8551725611256995,
#     'device_type': 'gpu',
#     'verbose': -1,
#     'random_state': 42
# }
# classifier_chain = make_pipeline(
#     StandardScaler(copy=False),
#     PCA(n_components=0.85, random_state=42, copy=False),
#     lgb.LGBMClassifier(**lgbm_params)
# )

# # Cross-validation
# print(f"Training split: X={X_train_cv.shape}, y={y_train_cv.shape}")
# print(f"Validation split: X={X_val_cv.shape}, y={y_val_cv.shape}")
# print("\nTraining Classifier on the combined samples...")
# classifier_chain.fit(X_train_cv, y_train_cv)
# print("Training finished.")

print("\nTraining Size: ", X_loaded.shape)
print("\nTraining Classifier on the all samples...")
classifier_chain.fit(X_loaded, y_loaded)
print("Training finished.")



Training Size:  (200000, 7068)

Training Classifier on the all samples...


/home/vizlab/Downloads/.venv/lib/python3.12/site-packages/sklearnex/decomposition/pca.py:345: UserWarning: Sklearnex always uses `covariance_eigh` solver instead of `full` when `svd_solver` parameter is set to `auto` for performance purposes.
  warn(
/home/vizlab/Downloads/.venv/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)


Training finished.


### 4. Test Predictions and Submission
Loading the testing feature matrix, generating predictions for the `test.csv` blind set, and standardizing the output for submisison.

In [9]:
# Predict on the new images
import pandas as pd
import datetime

print("Loading test features...")
test_variants = ["pre_X_test"]
X_test_final = load_cached_feature_matrix(test_variants, selected_features)
print(f"Test features loaded: {X_test_final.shape}")

print("Predicting test data labels...")
test_predictions = classifier_chain.predict(X_test_final)

# Create the submission dataframe
test_csv_path = "./test.csv"
submission_df = pd.read_csv(test_csv_path)

submission_df["label"] = test_predictions
output_filename = f"test_predictions_{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}.csv"
submission_df.to_csv(output_filename, index=False)

print(f"Predictions successfully saved to {output_filename}")


# ORIGINAL CODE (Validation diagnostics):
# from sklearn.metrics import classification_report, ConfusionMatrixDisplay
# import matplotlib.pyplot as plt

# print("Running Diagnostics...\n")

# # Score validation data
# val_predictions = classifier_chain.predict(X_val_cv)

# # Evaluate classification precision
# print(classification_report(y_val_cv, val_predictions, digits=4))

# # Run sub-train check for overfitting metrics, "shift_up_4"
# subset_preds = classifier_chain.predict(X_train_cv[:1500])
# train_acc = (y_train_cv[:1500] == subset_preds).mean()
# print(f"\nSub-Train Accuracy (Checking Overfitting): {train_acc:.4f}\n")

# # Render standardized Confusion Matrix
# ConfusionMatrixDisplay.from_predictions(y_val_cv, val_predictions, cmap="viridis")
# plt.title("Support Vector Machine - Validation Matrix")
# plt.show()


Loading test features...
Feature hog_9_(4, 4)_(2, 2)_2 missing for pre_X_test. Generating on the fly...


Generating hog_9_(4, 4)_(2, 2)_2 for pre_X_test: 100%|██████████| 10000/10000 [00:03<00:00, 2504.96it/s]


Feature hog_9_(6, 6)_(2, 2)_2 missing for pre_X_test. Generating on the fly...


Generating hog_9_(6, 6)_(2, 2)_2 for pre_X_test: 100%|██████████| 10000/10000 [00:02<00:00, 4522.46it/s]


Feature hog_9_(8, 8)_(2, 2)_2 missing for pre_X_test. Generating on the fly...


...

Feature color_hist missing for pre_X_test. Generating on the fly...


Generating color_hist for pre_X_test: 100%|██████████| 10000/10000 [00:00<00:00, 113686.80it/s]


Feature raw_color missing for pre_X_test. Generating on the fly...


Generating raw_color for pre_X_test: 100%|██████████| 10000/10000 [00:00<00:00, 321782.33it/s]


Test features loaded: (10000, 7068)
Predicting test data labels...
Predictions successfully saved to test_predictions_20260411170628.csv
